In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv('C:\\Users\\33142\\.cache\\kagglehub\\datasets\\blastchar\\telco-customer-churn\\versions\\1\\WA_Fn-UseC_-Telco-Customer-Churn.csv')
 

In [3]:
print(f"missing values:\n{df.isnull().sum()}")
print(f"\nduplicate values:\n{df.duplicated().sum()}")
print(f"\ndata type:\n{df.dtypes}")

missing values:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

duplicate values:
0

data type:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object


In [4]:
df = df.drop(['customerID'], axis = 1)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors = 'coerce')
df = df.dropna()

In [5]:
X =df.drop('Churn', axis = 1)
y= (df['Churn'] == 'Yes').astype(int)
print(f"The shape of X:{X.shape}")

The shape of X:(7032, 19)


In [6]:
categorical_cols = X.select_dtypes(include = ['object']).columns.tolist()
numerical_cols = X.select_dtypes(include = ['int64', 'float64']).columns.tolist()
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown = 'ignore', sparse_output = False) , categorical_cols)
         ])
X_processed = preprocessor.fit_transform(X)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y.values, test_size = 0.2, random_state = 42, stratify = y
)
print(f"X_train:{X_train.shape}, X_test:{X_test.shape}")

X_train:(5625, 45), X_test:(1407, 45)


In [8]:
class ChurnDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
        
train_dataset = ChurnDataset(X_train, y_train)
test_dataset = ChurnDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size = 64, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 64, shuffle = False)

In [9]:
class SimpleMLP(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.layers(x)

input_size = X_train.shape[1]
model = SimpleMLP(input_size)

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.003)

In [11]:
for epoch in range(60):
    model.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1:2d} | Loss:{total_loss/len(train_loader):.4f}")

Epoch  5 | Loss:0.4180
Epoch 10 | Loss:0.4114
Epoch 15 | Loss:0.4014
Epoch 20 | Loss:0.3940
Epoch 25 | Loss:0.3858
Epoch 30 | Loss:0.3749
Epoch 35 | Loss:0.3667
Epoch 40 | Loss:0.3572
Epoch 45 | Loss:0.3528
Epoch 50 | Loss:0.3432
Epoch 55 | Loss:0.3403
Epoch 60 | Loss:0.3234


In [12]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        outputs = model(batch_X)
        _, predicted = torch.max(outputs, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
accuracy = correct / total * 100
print(f"\naccuracy of the test sets:{accuracy:.2f}%")


accuracy of the test sets:78.39%
